1. 获取大模型 (Setup LLM)

In [4]:
# 导入dotenv 库的 load_dotenv 函数,用于加载环境变量文件(.env)中的配置
import dotenv
from dotenv.parser import parse_value
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import os

from openai import embeddings
from websockets.http11 import USER_AGENT

dotenv.load_dotenv()#读取当前加载目录下的.env文化
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY1')
os.environ['OPENAI_BASE_URL'] = os.getenv('OPENAI_BASE_URL1')
#创建大模型实例
llm=ChatOpenAI(model="deepseek-v4-flash")
#直接提供问题并调用llm
response=llm.invoke("什么的大模型")#默认使用deepseek-v4-flash
print(response)

content='“大模型”通常指**大型语言模型（Large Language Model, LLM）**，是人工智能领域的一种深度学习模型。它的“大”主要体现在三个方面：\n\n1. **庞大的参数规模**：通常拥有数十亿到数万亿个参数（如GPT-3有1750亿参数，GPT-4据估计达1.8万亿）。\n2. **海量的训练数据**：从互联网、书籍、论文等来源中学习，覆盖多种语言和知识领域。\n3. **强大的计算资源**：需要数千甚至上万张高性能GPU（如NVIDIA A100/H100）进行数周至数月的训练。\n\n**常见的大模型类型**：\n- **语言模型**（如GPT-4、Claude、文心一言、DeepSeek）——擅长对话、写作、翻译、代码生成。\n- **多模态模型**（如GPT-4V、Gemini）——能理解文本、图像、音频甚至视频。\n- **视觉模型**（如DALL·E 3、Stable Diffusion）——根据文字生成图像。\n- **科学领域模型**（如AlphaFold）——预测蛋白质结构。\n\n**核心能力**：可以通过理解上下文，进行推理、创作、问答、翻译等任务，甚至表现出“涌现能力”（即规模大到一定程度后自动产生的复杂推理、规划等能力，如解决数学题或编写复杂程序）。\n\n**局限性**：可能产生“幻觉”（编造不存在的知识）、有偏见、计算成本极高、无法真正理解情感或常识。\n\n如果你想了解**特定的大模型**（比如某一家公司的产品），或者想知道如何选择适合自己的模型，可以告诉我具体方向，我会进一步解释。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 480, 'prompt_tokens': 7, 'total_tokens': 487, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 132, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': 

2. 使用提示词模板 (Prompt Template)

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
#需要注意的是:这里需要指明具体的role,在这里是system和用户
prompt=ChatPromptTemplate.from_messages(
    [("system","你是世界级的技术文档编写者"),
     ("user","{input}")]#{input}为变量
)
#我们可以把prompt和具体的llm的调用和在一起
chain=prompt|llm
message=chain.invoke({"input":"大模型中的LangChain是什么?"})
print(message)
#print(type(message))


content='**LangChain 是什么？**  \nLangChain 是一个面向大语言模型（LLM）应用的开发框架，旨在简化、标准化并增强基于 LLM 的复杂工作流构建过程。它提供了模块化的抽象层，使开发者能够轻松地将 LLM 与外部数据源、工具、记忆系统及其他推理组件集成，从而构建出具有上下文感知、多步骤推理和工具调用能力的智能应用。\n\n---\n\n## 核心架构与设计理念\n\nLangChain 的核心设计围绕以下抽象展开，每个抽象都解决 LLM 应用开发中的特定痛点：\n\n| 抽象组件 | 职责与作用 |\n|--------|--------|\n| **Model I/O** | 统一管理 LLM 的调用，支持多模型（OpenAI、Hugging Face、本地模型等）、提示模板（Prompt Template）的生成与输出解析（Output Parser）。|\n| **Data Connection** | 提供文档加载器（Document Loader）、文本分割器（Text Splitter）、嵌入模型（Embeddings）与向量存储（Vector Store）的接口，实现非结构化数据（PDF、网页、数据库等）的向量化索引与检索。|\n| **Chains** | 定义 LLM 调用的执行序列与数据流。包括简单的 LLMChain（单一提示-响应）和复杂的 `SequentialChain`、`RouterChain`、`LLMMathChain` 等。支持将多个步骤组合为可复用的管道。|\n| **Agents** | 赋予 LLM 自主决策能力：解析用户意图，选择并调用工具（Tools），循环执行直到得到最终答案。Agent 通常结合 `ReAct`、`Plan-and-Solve` 等推理策略。|\n| **Memory** | 维护对话历史或上下文状态。支持 `ConversationBufferMemory`、`ConversationSummaryMemory`、`VectorStoreRetrieverMemory` 等，实现持久化的短时与长时记忆。|\n| **Callbacks** | 事件监听与日志、监控机制。可用于记录 LLM 调用耗时、Token 消耗、中间结果，便于调试与生产环境监控。|\n\n--

3. 使用输出解释器 (Output Parser)

In [6]:
#output_paper=StrOutPaper()
output_parser=JsonOutputParser()
#将其添加到上一个链中
#chain=prompt|llm
chain=prompt|llm|output_parser
#调用它并输出相同的问题,答案是一个字符串,而不是ChatMessage
#chain.invoke("input":"LangChain是什么")
chain.invoke({"input":"LangChain是什么?用JSON格式回复,问题用question,回答用answer"})

{'question': 'LangChain是什么?',
 'answer': 'LangChain是一个开源框架，用于构建基于大语言模型（LLM）的应用程序。它提供了一套模块化的工具和抽象，帮助开发者将LLM与外部数据源、API、记忆系统等集成，实现复杂的链式调用和智能代理功能，常见于聊天机器人、文档问答、自动化工作流等场景。'}

4. 使用向量存储 (Vector Store)

In [7]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_excel(r"E:\虎丘园林\第二组虎丘语料.xlsx")
docs = []
for _, row in df.iterrows():
    content = " | ".join([f"{col}: {row[col]}" for col in df.columns])
    docs.append(Document(page_content=content, metadata={"source": "虎丘语料.xlsx"}))
print(f"加载到 {len(docs)} 条数据")

# 使用轻量本地嵌入模型
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(f"切分后得到 {len(documents)} 个片段")

# 空列表保护，避免 FAISS.from_documents 崩溃
if not documents:
    print("⚠️ 没有获取到文档内容")
else:
    vectors = FAISS.from_documents(documents, embeddings)
    print("✅ 向量存储创建成功")


加载到 783 条数据


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4656.22it/s]


切分后得到 1025 个片段
✅ 向量存储创建成功


5. RAG (检索增强生成/Retrieval-Augmented Generation)

In [8]:
from langchain_core.prompts import PromptTemplate

# 创建检索器
retriever = vectors.as_retriever()
retriever.search_kwargs = {"k": 3}
docs = retriever.invoke("未来会发生什么")
retrieved_text = "\n\n".join([doc.page_content for doc in docs])

# 定义提示词模板并格式化
prompt_template = """
你是一个回答机器人。我的任务是根据下述给定的已知信息回答用户问题，
确保你的回复完全按照下面的已知信息，不编造答案。如果下述已知信息不足以回答用户的问题，
请直接回复"我无法回答你的问题"。
已知信息：
{info}
用户问：{question}
请用中文回答用户问题"""

prompt = PromptTemplate.from_template(prompt_template).format(
    info=retrieved_text,
    question="苏州园林有什么特点"
)

# 调用 LLM
response = llm.invoke(prompt)
print(response.content)

我无法回答你的问题


6. 使用Agent (Agent)

In [10]:
from langchain.agents import create_agent

# 创建检索工具（普通函数即可，不需要 @tool 装饰器）
def search_emotion(query: str) -> str:
    """搜索有关我情绪的语段"""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

# 创建 Agent（LangChain v1.x 新版 API）
agent = create_agent(
    model="openai:deepseek-v4-flash",
    tools=[search_emotion],
    system_prompt="你是一个有帮助的助手，可以根据检索到的内容回答用户问题，不要使用任何markdown格式，用纯文本回复",
)

# 运行
result = agent.invoke({
    "messages": [{"role": "user", "content": "虎丘园林有什么特点？"}]
})
print(result["messages"][-1].content_blocks[0]["text"])

虎丘园林，严格来说虎丘是苏州一个以自然山水和古迹为主的风景名胜区，并非典型意义上的私家园林，但它有着鲜明的特点。第一，它享有"吴中第一名胜"的美誉，宋代大文豪苏东坡曾说"到苏州不游虎丘，乃憾事也"，可见其地位之高。第二，虎丘历史悠久，最早可追溯到春秋时期，相传吴王夫差葬父阖闾于此，葬后三日有白虎蹲踞其上，因而得名"虎丘"。第三，虎丘的核心标志是虎丘塔，即云岩寺塔，这是一座千年古塔，塔身倾斜但依然屹立，成为苏州的标志性景观之一。第四，剑池是虎丘的另一大奇观，传说与吴王阖闾的墓葬有关，池水幽深，充满神秘色彩。第五，虎丘的特点是自然山水与人文古迹高度融合，岩壑奇峭、林泉清幽，既有山林野趣，又有厚重的历史文化底蕴，与苏州的宅第园林风格迥异，更偏向于自然山水园林的格局。
